# DSC 450 Project 2: EIA-930 ERCOT Data Wrangling

| | |
|---|---|
| **Title** | DSC 450 Project 2: EIA-930 ERCOT Data Wrangling |
| **Author** | Hali Sarah Parsons |
| **Role** | Data Wrangler |
| **Date** | 2026-10-10 |
| **Modified By** | Claude (2026-10-11: observations added, spike threshold raised, re-run safety) |
| **Description** | Reviews, cleans, and engineers features on hourly ERCOT grid data from EIA Form 930, and saves a modeling-ready file for the Data Scientist. |

**Project:** Forecast Regional Electricity Grid Demand  
**Input:** `eia930_erco_hourly.csv` (from `DSC450_Project2_EIA930_Data_Pull.ipynb`)  
**Output:** `eia930_erco_cleaned.csv`

### Project Goal
This notebook documents the exploration, cleaning, and feature engineering performed on the ERCOT hourly data. The final dataset is prepared for day-ahead demand forecasting.

**In Colab:** upload `eia930_erco_hourly.csv` with the Files panel (folder icon) before running.

## 1. Import Libraries

In [1]:
# Title: DSC 450 Project 2 - EIA-930 ERCOT Data Wrangling
# Author: Hali Sarah Parsons
# Date: 2026-10-10
# Modified By: Claude (2026-10-11)
# Description: Cleans hourly ERCOT grid data and builds forecasting features.

import numpy as np
import pandas as pd
# WHY this calendar: holidays change demand like a weekend does, and pandas
# ships the federal list so no extra install is needed.
from pandas.tseries.holiday import USFederalHolidayCalendar

## 2. Cleaning Log

**Why keep a log:** the Data Wrangling reading stresses that cleaning is iterative and that documentation is what lets you trace and reverse a decision later. Every change below is recorded here, and the finished log goes in the report appendix.

In [2]:
cleaning_log = []

def log_step(step, rows_affected, reason):
    '''Record one cleaning decision.

    WHY a function: every step is logged the same way, so the final table
    is complete and nothing depends on remembering to write it down.
    '''
    cleaning_log.append({'step': step, 'rows_affected': int(rows_affected), 'reason': reason})
    print(f'{step}: {int(rows_affected):,} rows')

## 3. Load the Dataset

**Why UTC is the index:** UTC has no daylight-saving jumps, so every hour appears exactly once. Central time is added later for features, because people's routines follow the local clock.

In [3]:
df = pd.read_csv('eia930_erco_hourly.csv')

# WHY utc=True: the saved timestamps carry a +00:00 offset, and forcing UTC
# guarantees a timezone-aware index even if a row was written differently.
df['period_utc'] = pd.to_datetime(df['period_utc'], utc=True)
df = df.set_index('period_utc').sort_index()

# WHY drop: the saved Central column is text; it is rebuilt from the index in
# Section 11 so the two can never disagree.
df = df.drop(columns=['period_central'], errors='ignore')

## 4. Dataset Overview

The structure, dimensions, and data types are examined before any changes are made.

In [4]:
df.shape

(41616, 4)

In [5]:
df.head()

,demand_mwh,demand_forecast_mwh,net_generation_mwh,interchange_mwh
period_utc,,,,
2022-01-01 00:00:00+00:00,45943.0,44669.0,45659.0,-284.0
2022-01-01 01:00:00+00:00,46427.0,44468.0,46255.0,-172.0
2022-01-01 02:00:00+00:00,45058.0,42466.0,44985.0,-73.0
2022-01-01 03:00:00+00:00,43430.0,40562.0,43449.0,19.0
2022-01-01 04:00:00+00:00,42058.0,39027.0,42136.0,79.0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 41616 entries, 2022-01-01 00:00:00+00:00 to 2026-09-30 23:00:00+00:00
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   demand_mwh           41616 non-null  float64
 1   demand_forecast_mwh  41496 non-null  float64
 2   net_generation_mwh   41592 non-null  float64
 3   interchange_mwh      41489 non-null  float64
dtypes: float64(4)
memory usage: 1.6 MB


In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
demand_mwh,41616.0,53422.986976,11574.551836,31949.0,44748.00,50665.0,60380.25,91075.0
demand_forecast_mwh,41496.0,53669.938404,11781.078490,30412.0,44948.00,50948.0,60680.00,92091.0
net_generation_mwh,41592.0,53353.852183,11548.116642,31806.0,44707.75,50593.0,60261.50,91117.0
interchange_mwh,41489.0,-70.368146,326.627356,-1193.0,-219.00,-36.0,110.00,1162.0


### Observations

The dataset holds 41,616 hourly rows and four numeric series. It covers January 1, 2022 through September 30, 2026 in UTC. All values are in megawatthours and stored as decimal numbers.

| Series | Minimum | Median | Mean | Maximum |
|---|---|---|---|---|
| Demand | 31,949 | 50,665 | 53,423 | 91,075 |
| ERCOT demand forecast | 30,412 | 50,948 | 53,670 | 92,091 |
| Net generation | 31,806 | 50,593 | 53,354 | 91,117 |
| Interchange | -1,193 | -36 | -70 | 1,162 |

- **Demand has a wide range.** The highest hour (91,075 MWh) is almost three times the lowest (31,949 MWh). The standard deviation is 11,575 MWh.
- **Demand is skewed toward high values.** The mean (53,423) sits above the median (50,665), which fits a long tail of high-demand hours.
- **The ERCOT forecast tracks demand closely.** Its mean is about 250 MWh above the demand mean, less than half a percent.
- **Net generation nearly equals demand.** The two means differ by about 70 MWh, which matches the average interchange of -70 MWh.
- **Interchange is small.** It never exceeds about 1,200 MWh in either direction, roughly 2 percent of average demand. ERCOT meets almost all of its own load.
- **Demand is complete.** It has 41,616 non-null values. The forecast has 41,496, net generation 41,592, and interchange 41,489 (Section 8).

## 5. Data Integrity Review

The Data Wrangling reading lists characteristics of a high-integrity dataset. This is how the EIA-930 data measures up before cleaning.

| Characteristic | Assessment |
|---|---|
| Known provenance | Reported by ERCOT to the U.S. Energy Information Administration on Form EIA-930. Public domain. |
| Well-annotated | EIA publishes definitions for each series (D, DF, NG, TI). |
| Timely | Updated hourly. The most recent days can still be revised by the operator. |
| Complete | Demand is complete: no missing hours and no missing values. The forecast, net generation, and interchange have small gaps (Section 8). |
| High volume | 41,616 hourly rows from January 2022 through September 2026. |
| Multivariate | Four series per hour. No weather, which limits what the model can explain. |
| Atomic | One row per hour, the finest grain EIA collects. |
| Consistent | All values are in megawatthours. No impossible values were found (Section 9). |
| Clear | Column names were made readable in the data pull notebook. |
| Dimensionally structured | Not yet. Calendar dimensions are added in Section 11. |

## 6. Duplicate Records Assessment

**Why it matters:** a time-series model expects each hour once. A repeated hour would count that hour's demand twice in any average or lag.

In [8]:
dup_mask = df.index.duplicated(keep='first')
log_step('Drop duplicate timestamps', dup_mask.sum(), 'Each hour must appear once.')
# WHY keep='first': the pull was sorted by time, so the first row is the one
# the API returned first; later repeats add no information.
df = df[~dup_mask]

Drop duplicate timestamps: 0 rows


### Observations

No duplicate timestamps were found. Each hour appears once.

## 7. Missing Hours

**Why this is separate from null values:** an hour can be missing as a whole row, which `isnull()` cannot see. Rebuilding the full hourly range exposes those gaps as empty rows so they can be counted and filled.

In [9]:
full_range = pd.date_range(df.index.min(), df.index.max(), freq='h')
missing_hours = full_range.difference(df.index)

# WHY reindex: lag features count rows, so a missing row would silently shift
# "24 hours ago" to the wrong hour.
df = df.reindex(full_range)
df.index.name = 'period_utc'
# WHY a flag: the Data Scientist can see which hours never existed in the source.
df['was_missing_hour'] = df.index.isin(missing_hours).astype(int)

log_step('Insert rows for missing hours', len(missing_hours), 'Restore an unbroken hourly sequence.')

Insert rows for missing hours: 0 rows


### Observations

No hours are missing. The 41,616 rows match the full hourly range exactly, so the sequence was already unbroken.

## 8. Missing Value Exploration

In [10]:
df.isnull().sum()

,0
demand_mwh,0
demand_forecast_mwh,120
net_generation_mwh,24
interchange_mwh,127
was_missing_hour,0


In [11]:
# WHY group by year: shows whether gaps are spread out or concentrated in one
# period, which would point to a reporting problem instead of random loss.
df.isnull().groupby(df.index.year).sum()

,demand_mwh,demand_forecast_mwh,net_generation_mwh,interchange_mwh,was_missing_hour
period_utc,,,,,
2022,0,0,0,0,0
2023,0,0,0,6,0
2024,0,0,0,1,0
2025,0,48,24,48,0
2026,0,72,0,72,0


In [12]:
# WHY list the dates: whole missing days suggest a reporting outage, while
# scattered single hours suggest isolated dropouts. The dates tell them apart.
gap_cols = ['demand_forecast_mwh', 'net_generation_mwh', 'interchange_mwh']
gaps_by_date = df[gap_cols].isnull().groupby(df.index.date).sum()
gaps_by_date[gaps_by_date.sum(axis=1) > 0]

,demand_forecast_mwh,net_generation_mwh,interchange_mwh
2023-04-13,0,0,1
2023-10-04,0,0,1
2023-10-12,0,0,2
2023-11-07,0,0,1
2023-11-28,0,0,1
2024-03-28,0,0,1
2025-12-04,0,0,17
2025-12-05,17,17,24
2025-12-06,24,7,7
2025-12-07,7,0,0


### Observations

Demand is complete in every year.

The other three series have small gaps:

| Series | Missing hours | Share of rows | Where |
|---|---|---|---|
| ERCOT demand forecast | 120 | 0.29% | 48 in 2025, 72 in 2026 |
| Net generation | 24 | 0.06% | All in 2025 |
| Interchange | 127 | 0.31% | 6 in 2023, 1 in 2024, 48 in 2025, 72 in 2026 |

The dates show two different patterns:

- **Scattered single hours (interchange only).** Seven hours across six dates: April 13, October 4, October 12 (2 hours), November 7, and November 28, 2023, and March 28, 2024.
- **Reporting outages.** Four episodes account for every other gap:

| Episode (UTC dates) | Interchange | Forecast | Net generation |
|---|---|---|---|
| December 4 to 7, 2025 | 48 hours | 48 hours | 24 hours |
| August 5 to 7, 2026 | 24 hours | 24 hours | none |
| August 8 to 10, 2026 | 24 hours | 24 hours | none |
| September 3 to 5, 2026 | 24 hours | 24 hours | none |

In each episode the forecast gap falls one day after the interchange gap. Each block totals 24 or 48 hours split across UTC dates, which is consistent with whole Central-time days going unreported.

These gaps stay as missing values because those columns are not the model target (Section 10). Demand was reported through all four episodes.

## 9. Impossible Values

**Why these rules:** a live grid always has load and generation, so zero or negative demand, forecast, or net generation is a reporting error, not a real reading. Interchange is left alone because it is negative whenever ERCOT imports power.

**Why set to missing instead of deleting the row:** deleting would break the hourly sequence rebuilt in Section 7. Marking the value missing lets Section 10 fill it.

In [13]:
# WHY keep the original: the untouched demand stays available for auditing.
df['demand_raw_mwh'] = df['demand_mwh']

for col in ['demand_mwh', 'demand_forecast_mwh', 'net_generation_mwh']:
    bad = df[col] <= 0
    log_step(f'Set {col} <= 0 to missing', bad.sum(), 'A live grid cannot report zero or negative here.')
    df.loc[bad, col] = np.nan

Set demand_mwh <= 0 to missing: 0 rows
Set demand_forecast_mwh <= 0 to missing: 0 rows
Set net_generation_mwh <= 0 to missing: 0 rows


### Spike detection

**Why a rolling median:** demand changes with the hour and the season, so one fixed cutoff would either miss errors in winter or flag real summer peaks. Comparing each hour to the median of the two days around it adapts to both.

**Why a high threshold (8):** heat waves, winter storms, and deep overnight lows are real, and they are exactly what a demand forecast must handle. The goal is to catch gross reporting errors only. The first run used 6 and flagged two genuine overnight lows (see Observations), so the threshold was raised.

In [14]:
SPIKE_WINDOW = 49      # hours: one day on each side of the hour being checked
# WHY 8: at 6 the rule flagged two real overnight lows on 2023-03-29.
SPIKE_THRESHOLD = 8

d = df['demand_mwh']
# WHY center=True: uses hours before and after, so a spike is judged against
# its neighbors on both sides.
rolling_median = d.rolling(SPIKE_WINDOW, center=True, min_periods=12).median()
abs_dev = (d - rolling_median).abs()
# WHY median absolute deviation: unlike standard deviation, it is not inflated
# by the very spikes being searched for.
mad = abs_dev.rolling(SPIKE_WINDOW, center=True, min_periods=12).median()
# WHY replace 0: a flat stretch has zero spread and would divide by zero.
# WHY 1.4826: scales MAD to be comparable to a standard deviation.
robust_z = abs_dev / (1.4826 * mad.replace(0, np.nan))

spike_mask = robust_z > SPIKE_THRESHOLD
df.loc[spike_mask, ['demand_raw_mwh']].assign(
    rolling_median=rolling_median[spike_mask], robust_z=robust_z[spike_mask]
)

,demand_raw_mwh,rolling_median,robust_z
period_utc,,,


In [15]:
log_step('Set demand spikes to missing', spike_mask.sum(), 'Value is far outside the surrounding two days.')
df.loc[spike_mask, 'demand_mwh'] = np.nan

Set demand spikes to missing: 0 rows


### Observations

No zero or negative values were found in demand, forecast, or net generation.

At the first threshold of 6, the spike rule flagged two hours: March 29, 2023 at 08:00 and 09:00 UTC, which is 3 and 4 a.m. Central. Demand was 34,740 and 34,569 MWh against a two-day median of 42,711 MWh, about 19 percent lower. The two readings sit side by side at the lowest point of the night and agree with each other. That pattern fits a real overnight low on a mild spring night, not a reporting error.

The threshold was raised to 8 so these hours stay in the data. At 8 the rule flags nothing, and no demand values were removed.

## 10. Filling Demand Gaps

**Why two methods:**
- **Gaps of 3 hours or less:** demand moves smoothly from hour to hour, so a straight line between the neighbors is a close estimate.
- **Longer gaps:** a straight line would flatten the daily peak. The same hour one week earlier keeps both the time-of-day and day-of-week shape.

**Why only demand is filled:** it is the target the model learns from and needs an unbroken series. The ERCOT forecast is the benchmark we compare against, so inventing values for it would bias that comparison. Net generation and interchange are context columns and stay as reported.

In [16]:
SHORT_GAP_HOURS = 3

demand = df['demand_mwh'].copy()
is_gap = demand.isna()
# WHY cumsum on changes: gives every run of consecutive missing hours its own
# id, so the length of each gap can be measured.
gap_id = (is_gap != is_gap.shift()).cumsum()
gap_length = is_gap.groupby(gap_id).transform('sum')
short_gap = is_gap & (gap_length <= SHORT_GAP_HOURS)

# WHY limit_area='inside': never extrapolate past the first or last real value.
interpolated = demand.interpolate(method='time', limit_area='inside')
demand[short_gap] = interpolated[short_gap]

# WHY shift(168): 168 hours is one week, so this is the same hour and weekday.
# WHY the backward fill second: covers gaps in the first week, which have no
# earlier week to borrow from.
demand = demand.fillna(demand.shift(168)).fillna(demand.shift(-168))

# WHY combine with the existing flag: if this cell is run twice, the second run
# finds no gaps and would otherwise reset every flag to 0.
already_flagged = df['demand_imputed'] == 1 if 'demand_imputed' in df else False
df['demand_imputed'] = ((is_gap & demand.notna()) | already_flagged).astype(int)
df['demand_mwh'] = demand

log_step('Fill short demand gaps by interpolation', (short_gap & demand.notna()).sum(), f'Gaps of {SHORT_GAP_HOURS} hours or less.')
log_step('Fill long demand gaps with same hour last week', (is_gap & ~short_gap & demand.notna()).sum(), 'Preserves daily and weekly shape.')
print('Demand still missing:', int(demand.isna().sum()))

Fill short demand gaps by interpolation: 0 rows
Fill long demand gaps with same hour last week: 0 rows
Demand still missing: 0


### Observations

No demand values needed filling. Every value in `demand_mwh` is the figure EIA reported, and `demand_imputed` is 0 for all rows. The fill logic stays in place so the notebook still works if a later data pull contains gaps.

## 11. Feature Engineering

New variables are created for the forecasting models.

### Calendar features

**Why Central time:** demand follows the local clock (the late-afternoon peak, the weekend dip). Using UTC hours would shift those patterns by five or six hours and move them again at each daylight-saving change.

In [17]:
central = df.index.tz_convert('America/Chicago')
df['period_central'] = central

df['hour'] = central.hour
df['day_of_week'] = central.dayofweek          # Monday = 0
df['is_weekend'] = (central.dayofweek >= 5).astype(int)
df['month'] = central.month
df['year'] = central.year

# WHY the same season map as Project 1: keeps the team's definitions consistent.
df['season'] = df['month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall', 10: 'Fall', 11: 'Fall',
})

# WHY tz_localize(None) then normalize: the holiday list is plain dates, so the
# local timestamp is reduced to its local calendar date before matching.
local_dates = central.tz_localize(None).normalize()
holidays = USFederalHolidayCalendar().holidays(start=local_dates.min(), end=local_dates.max())
df['is_holiday'] = local_dates.isin(holidays).astype(int)

### Lag features

**Why lags:** the best single predictor of demand at a given hour is demand at the same hour yesterday and last week.

**Why nothing more recent than 24 hours:** this is a day-ahead forecast. At the time the forecast is made, the most recent 24 hours have not happened yet, so using them would leak the future into the model and make it look better than it is.

In [18]:
df['demand_lag_24'] = df['demand_mwh'].shift(24)
df['demand_lag_168'] = df['demand_mwh'].shift(168)
# WHY shift before rolling: the average covers the 24 hours that end a full day
# before the target hour, which respects the day-ahead rule above.
df['demand_roll_mean_24'] = df['demand_mwh'].shift(24).rolling(24).mean()

### ERCOT forecast error

**Why:** this is the benchmark the team's model has to beat. It uses reported demand only, so imputed hours do not count for or against ERCOT.

In [19]:
# WHY mask imputed rows: an estimate is not a fair truth to score ERCOT against.
actual_reported = df['demand_mwh'].where(df['demand_imputed'] == 0)
df['ercot_forecast_error_mwh'] = df['demand_forecast_mwh'] - actual_reported

## 12. Data Quality Validation

The cleaned dataset is reviewed to confirm that cleaning produced consistent, analysis-ready data.

In [20]:
expected_rows = len(pd.date_range(df.index.min(), df.index.max(), freq='h'))
print('Dataset shape:', df.shape)
print('Range:', df.index.min(), 'to', df.index.max())
print('Rows match full hourly range:', len(df) == expected_rows)
print('Duplicate timestamps:', int(df.index.duplicated().sum()))
print('Demand missing:', int(df['demand_mwh'].isna().sum()))
print('Demand <= 0:', int((df['demand_mwh'] <= 0).sum()))
print(f"Demand imputed: {int(df['demand_imputed'].sum()):,} ({df['demand_imputed'].mean():.2%})")
print('Season missing:', int(df['season'].isna().sum()))

Dataset shape: (41616, 19)
Range: 2022-01-01 00:00:00+00:00 to 2026-09-30 23:00:00+00:00
Rows match full hourly range: True
Duplicate timestamps: 0
Demand missing: 0
Demand <= 0: 0
Demand imputed: 0 (0.00%)
Season missing: 0


In [ ]:
# WHY a table: this is the cleaning record for the report appendix.
pd.DataFrame(cleaning_log)

### Observations

The cleaned dataset has 41,616 rows and 19 columns: the 4 original series plus 15 added columns (3 audit columns, 8 calendar features, 3 lag features, and the ERCOT forecast error). The row count matches the full hourly range. There are no duplicate timestamps, no missing or non-positive demand values, and no imputed hours. The cleaning log shows zero rows changed at every step, so the demand series is delivered exactly as EIA reported it.

## 13. Save Cleaned/Engineered Dataset

**Note for the Data Scientist:** the first 168 rows have empty lag features because there is no earlier week to look back to. Drop them before training.

### Data dictionary

| Column | Meaning |
|---|---|
| `period_utc` | Hour of the observation, UTC |
| `period_central` | Same hour in Central time |
| `demand_mwh` | Cleaned hourly demand (target), megawatthours |
| `demand_raw_mwh` | Demand exactly as reported by EIA |
| `demand_imputed` | 1 if `demand_mwh` was filled in by this notebook |
| `was_missing_hour` | 1 if the hour was absent from the source data |
| `demand_forecast_mwh` | ERCOT day-ahead demand forecast |
| `net_generation_mwh` | Total hourly generation |
| `interchange_mwh` | Net flow to neighboring regions (negative = imports) |
| `hour`, `day_of_week`, `is_weekend`, `month`, `year`, `season`, `is_holiday` | Calendar features in Central time |
| `demand_lag_24`, `demand_lag_168` | Demand 24 and 168 hours earlier |
| `demand_roll_mean_24` | Mean demand over the 24 hours ending one day earlier |
| `ercot_forecast_error_mwh` | ERCOT forecast minus reported demand |

In [21]:
df.to_csv('eia930_erco_cleaned.csv')

## 14. Conclusion

The EIA-930 ERCOT data arrived in very good condition. All 41,616 hours from January 2022 through September 2026 are present, with no duplicates and no missing or impossible demand values. No demand value was changed or filled.

**Data profile**
- Demand ranges from 31,949 to 91,075 MWh, with a mean of 53,423 and a median of 50,665.
- The ERCOT forecast averages about 250 MWh above actual demand, less than half a percent.
- Interchange stays within about 1,200 MWh either way, so ERCOT serves almost all of its own load.

**Decisions made**
- The spike rule was loosened from 6 to 8 after it flagged two real overnight lows on March 29, 2023.
- Gaps in the ERCOT forecast (120 hours), net generation (24 hours), and interchange (127 hours) were left as missing. Filling the forecast would bias the benchmark comparison. Almost all of these gaps fall in four reporting outages: December 4 to 7, 2025, August 5 to 7 and 8 to 10, 2026, and September 3 to 5, 2026.

**What was added**
- Central-time calendar features: hour, day of week, weekend, month, year, season, and federal holiday.
- Day-ahead-safe lag features: demand 24 and 168 hours earlier, and a 24-hour rolling mean ending one day earlier.
- ERCOT's forecast error, as the benchmark for the team's model.

**Limits to keep in mind**
- The data has no weather, which is the main driver of demand the models cannot see.
- The 120 hours without an ERCOT forecast (0.29% of the data) cannot be used in the benchmark comparison. All four outages fall inside the most recent 12 months, which is the planned test period.
- EIA can revise the most recent days, so a later pull may differ slightly at the end of the range.
- The first 168 rows have empty lag features and should be dropped before training.

**Handoff:** `eia930_erco_cleaned.csv` is ready for modeling.